# Consolidación SIPSA-DANE – Precios mayoristas (2013 – 2026)

Este notebook consolida todos los archivos Excel de la carpeta
`raw_data/Bases de Datos SIPSA-DANE Completo` en un único archivo CSV:

`processed_data/sipsa_precios_mayoristas_consolidado.csv`

**Consideraciones de los archivos fuente:**
- `series-historicas-precios-mayoristas 2013-2017.xlsx` trae un año por hoja (`1.1` = 2013 … `1.5` = 2017).
- 2019 – 2023 traen una hoja por mes; 2018 y 2024 – 2026 traen una sola hoja con todo el año.
- La fila de encabezado cambia de posición según el archivo, así que se detecta automáticamente (la fila cuya primera celda es `Fecha`).
- Los nombres de columnas cambian entre años (`Fuente` / `Mercado`, `Precio` / `Precio promedio por kilogramo*`), por lo que se estandarizan.
- Las hojas auxiliares (`Índice`, `CPC`) se omiten. La hoja `Fuentes` del archivo 2026 se usa como tabla de ubicación de los mercados.
- Solo se conservan los grupos **CARNES** y **GRANOS Y CEREALES**.
- A toda la serie histórica se le agrega `Departamento` y `Municipio` (con sus códigos DANE), que en los archivos originales solo vienen desde 2026.


## 1. Preparar el entorno

In [ ]:
import os
import sys

REPO_URL = "https://github.com/yevasa93/AnomaliasClimaticasArrozCarne.git"
REPO_DIR = "AnomaliasClimaticasArrozCarne"

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # En Colab se clona el repositorio público (o se actualiza si ya existe)
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    else:
        !git -C {REPO_DIR} pull
    os.chdir(REPO_DIR)

print("Directorio de trabajo:", os.getcwd())

In [ ]:
import glob
import unicodedata

import pandas as pd

RAW_DIR = os.path.join("raw_data", "Bases de Datos SIPSA-DANE Completo")
OUT_DIR = "processed_data"
OUT_FILE = os.path.join(OUT_DIR, "sipsa_precios_mayoristas_consolidado.csv")

os.makedirs(OUT_DIR, exist_ok=True)

archivos = sorted(glob.glob(os.path.join(RAW_DIR, "*.xlsx")))
print(f"{len(archivos)} archivos encontrados:")
for a in archivos:
    print("  -", os.path.basename(a))

## 2. Funciones de lectura y estandarización

Cada hoja se lee sin encabezado, se ubica la fila donde la primera celda es `Fecha` y se toma como encabezado.
Luego se renombran las columnas a un esquema común:

| Columna final | Nombres originales |
|---|---|
| `Fecha` | Fecha |
| `Grupo` | Grupo |
| `Producto` | Producto |
| `Mercado` | Fuente / Mercado |
| `Precio_kg` | Precio / Precio por kilogramo* / Precio promedio por kilogramo* |

`Departamento` y `Municipio` solo vienen en el archivo 2026, por eso no se leen aquí: se asignan a toda la serie en la sección 6 a partir de la hoja `Fuentes`.

In [ ]:
COLUMNAS_FINALES = ["Fecha", "Grupo", "Producto", "Mercado", "Precio_kg"]


def normalizar_texto(texto):
    """Minúsculas, sin tildes y sin espacios repetidos (para comparar nombres de columnas)."""
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.lower().split())


def mapear_columna(nombre):
    n = normalizar_texto(nombre)
    if n == "fecha":
        return "Fecha"
    if n == "grupo":
        return "Grupo"
    if n == "producto":
        return "Producto"
    if n in ("fuente", "mercado"):
        return "Mercado"
    if n.startswith("precio"):
        return "Precio_kg"
    return None  # columna que no se conserva


def leer_hoja(df_crudo, archivo, hoja):
    """Recibe una hoja leída sin encabezado y devuelve un DataFrame estandarizado (o None)."""
    primera_col = df_crudo.iloc[:30, 0].astype(str).str.strip()
    filas_encabezado = primera_col.index[primera_col == "Fecha"]
    if len(filas_encabezado) == 0:
        return None  # hoja auxiliar (Índice, CPC, Fuentes...)

    fila = filas_encabezado[0]
    df = df_crudo.iloc[fila + 1:].copy()
    df.columns = [mapear_columna(c) for c in df_crudo.iloc[fila]]
    df = df.loc[:, [c is not None for c in df.columns]]

    faltantes = set(COLUMNAS_FINALES) - set(df.columns)
    if faltantes:
        raise ValueError(f"{archivo} / {hoja}: faltan columnas {faltantes}")

    df = df[COLUMNAS_FINALES]
    df["Archivo_origen"] = archivo
    df["Hoja_origen"] = str(hoja)
    return df

## 3. Lectura de todos los archivos y hojas

In [ ]:
partes = []
resumen = []
hoja_fuentes = None  # tabla Mercado -> Departamento / Municipio (archivo 2026)

for ruta in archivos:
    archivo = os.path.basename(ruta)
    hojas = pd.read_excel(ruta, sheet_name=None, header=None, dtype=object)  # todas las hojas
    for hoja, df_crudo in hojas.items():
        if hoja == "Fuentes":
            hoja_fuentes = df_crudo
        df = leer_hoja(df_crudo, archivo, hoja)
        if df is None:
            resumen.append((archivo, hoja, "omitida", 0))
            continue
        partes.append(df)
        resumen.append((archivo, hoja, "leída", len(df)))

resumen = pd.DataFrame(resumen, columns=["Archivo", "Hoja", "Estado", "Filas_crudas"])
datos = pd.concat(partes, ignore_index=True)

print(f"Hojas leídas: {(resumen.Estado == 'leída').sum()} | omitidas: {(resumen.Estado == 'omitida').sum()}")
print(f"Filas crudas consolidadas: {len(datos):,}")
resumen

## 4. Limpieza

- Conversión de `Fecha` a fecha y `Precio_kg` a número.
- Eliminación de filas vacías o notas al pie (sin fecha válida o sin precio).
- Limpieza de espacios en los textos.
- Eliminacin de registros duplicados.

In [ ]:
df = datos.copy()

df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
df["Precio_kg"] = pd.to_numeric(df["Precio_kg"], errors="coerce")

for col in ["Grupo", "Producto", "Mercado"]:
    df[col] = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

filas_antes = len(df)
df = df.dropna(subset=["Fecha", "Producto", "Mercado", "Precio_kg"])
print(f"Filas eliminadas por fecha/precio/producto/mercado vacío (notas al pie, filas en blanco): {filas_antes - len(df):,}")

# El grupo se unifica en mayúsculas y sin tildes
# (en algunos archivos aparece "TUBÉRCULOS, RAÍCES Y PLÁTANOS" y en otros "TUBERCULOS, RAICES Y PLATANOS")
df["Grupo"] = df["Grupo"].map(lambda g: normalizar_texto(g).upper()).astype("string")

filas_antes = len(df)
df = df.drop_duplicates(subset=["Fecha", "Grupo", "Producto", "Mercado", "Precio_kg"])
print(f"Duplicados eliminados: {filas_antes - len(df):,}")

df["Anio"] = df["Fecha"].dt.year
df["Mes"] = df["Fecha"].dt.month

print(f"Filas después de la limpieza (todos los grupos): {len(df):,}")
df["Grupo"].value_counts()

## 5. Filtro de grupos: CARNES y GRANOS Y CEREALES

In [ ]:
GRUPOS = ["CARNES", "GRANOS Y CEREALES"]

df = df[df["Grupo"].isin(GRUPOS)].copy()

print(f"Filas después del filtro: {len(df):,}")
df["Grupo"].value_counts()

## 6. Departamento y municipio para toda la serie historica

Los archivos originales solo traen la ubicación desde 2026. Para los años anteriores se usa la hoja `Fuentes`
del archivo 2026, que relaciona cada mercado con su departamento y municipio (con códigos DANE).

- El cruce se hace por nombre de mercado normalizado (sin tildes, mayúsculas ni signos de puntuación),
  así *"Bogotá, D.C., Corabastos"* y *"Bogota, D.C. Corabastos"* se consideran el mismo mercado.
- Algunos mercados históricos cambiaron de nombre o ya no se reportan en 2026; para ellos se define
  la equivalencia manualmente en `ALIAS_MERCADOS` (mismo municipio).

In [ ]:
import re


def clave_mercado(nombre):
    """Llave de cruce: sin tildes, en minúsculas y sin signos de puntuación."""
    texto = normalizar_texto(nombre)
    return " ".join(re.sub(r"[^a-z0-9]+", " ", texto).split())


def codigo_dane(valor, digitos):
    """Devuelve el código DANE como texto con ceros a la izquierda (p. ej. 5 -> '05')."""
    if pd.isna(valor):
        return pd.NA
    return str(int(float(valor))).zfill(digitos)


# Mercados históricos que no aparecen en la hoja Fuentes -> mercado equivalente del mismo municipio
ALIAS_MERCADOS = {
    "Pereira, La 41": "Pereira, La 41-Impala",
    "Cali, Santa Helena": "Cali, Santa Elena",
    "San Vicente (Antioquia)": "San Vicente Ferrer (Antioquia)",
    "Malambo, Atlantico": "Malambo (Atlántico), Carnes y carnes",
}

# Tabla de ubicación a partir de la hoja Fuentes (primera fila = encabezado)
ubicacion = hoja_fuentes.iloc[1:, :5].copy()
ubicacion.columns = ["Mercado_fuente", "Departamento", "Cod_departamento", "Municipio", "Cod_municipio"]
ubicacion = ubicacion.dropna(subset=["Mercado_fuente", "Departamento"])
ubicacion["Cod_departamento"] = ubicacion["Cod_departamento"].map(lambda v: codigo_dane(v, 2))
ubicacion["Cod_municipio"] = ubicacion["Cod_municipio"].map(lambda v: codigo_dane(v, 5))
for col in ["Departamento", "Municipio"]:
    ubicacion[col] = ubicacion[col].astype("string").str.strip()
ubicacion["clave"] = ubicacion["Mercado_fuente"].map(clave_mercado)
ubicacion = ubicacion.drop_duplicates("clave")
print(f"Mercados en la tabla de ubicación: {len(ubicacion)}")

# Cruce con la serie histórica
df["clave"] = df["Mercado"].map(lambda m: clave_mercado(ALIAS_MERCADOS.get(m, m)))
df = df.merge(
    ubicacion[["clave", "Departamento", "Cod_departamento", "Municipio", "Cod_municipio"]],
    on="clave",
    how="left",
).drop(columns="clave")

sin_ubicacion = df.loc[df["Departamento"].isna(), "Mercado"].value_counts()
print(f"Mercados sin ubicación: {len(sin_ubicacion)} ({sin_ubicacion.sum():,} filas)")
sin_ubicacion  # si aparece alguno, agregarlo a ALIAS_MERCADOS

In [ ]:
df = df[[
    "Fecha", "Anio", "Mes", "Grupo", "Producto", "Mercado",
    "Departamento", "Cod_departamento", "Municipio", "Cod_municipio",
    "Precio_kg", "Archivo_origen", "Hoja_origen",
]]
df = df.sort_values(["Fecha", "Grupo", "Producto", "Mercado"]).reset_index(drop=True)

print(f"Filas finales: {len(df):,}")
df.head()

## 7. Validaciones

In [ ]:
print("Rango de fechas:", df["Fecha"].min().date(), "a", df["Fecha"].max().date())

validacion = df.groupby("Anio").agg(
    registros=("Precio_kg", "size"),
    meses=("Mes", "nunique"),
    productos=("Producto", "nunique"),
    mercados=("Mercado", "nunique"),
    departamentos=("Departamento", "nunique"),
    municipios=("Municipio", "nunique"),
)
validacion

In [ ]:
# Registros por departamento
df.groupby(["Departamento", "Grupo"]).size().unstack(fill_value=0).sort_values("CARNES", ascending=False)

In [ ]:
# Productos por grupo
df.groupby(["Grupo", "Producto"]).size().sort_values(ascending=False).head(40)

> **Nota sobre 2026:** a partir de 2026 SIPSA incluye además precios en **molinos de arroz** (p. ej. *Espinal, Fedearroz*, *Ibagué, Molino Federal*), por eso ese año tiene más mercados que los anteriores.

## 8. Exportar CSV


In [ ]:
COLUMNAS_CSV = [
    "Fecha", "Anio", "Mes", "Grupo", "Producto", "Mercado",
    "Departamento", "Cod_departamento", "Municipio", "Cod_municipio", "Precio_kg",
]
df[COLUMNAS_CSV].to_csv(OUT_FILE, index=False, encoding="utf-8", date_format="%Y-%m-%d")

tamano_mb = os.path.getsize(OUT_FILE) / 1024 ** 2
print(f"Archivo generado: {OUT_FILE} ({tamano_mb:.1f} MB, {len(df):,} filas)")
if tamano_mb > 100:
    print("ADVERTENCIA: GitHub no acepta archivos mayores a 100 MB.")

In [ ]:
# En Colab: descargar el CSV generado para subirlo al repositorio
if EN_COLAB:
    from google.colab import files
    files.download(OUT_FILE)

## 9. Lectura del CSV desde el repositorio público de GitHub

lo generado siempre se dejo en el repositorio publico y se puede leer desde cualquier notebook así
(los códigos DANE se leen como texto para conservar los ceros a la izquierda):

In [ ]:
URL_CSV = (
    "https://raw.githubusercontent.com/yevasa93/AnomaliasClimaticasArrozCarne/main/"
    "processed_data/sipsa_precios_mayoristas_consolidado.csv"
)

sipsa = pd.read_csv(
    URL_CSV,
    parse_dates=["Fecha"],
    dtype={"Cod_departamento": str, "Cod_municipio": str},
)
print(sipsa.shape)
sipsa.head()